In [1]:
%run ./../notebook_init.py

import os
import uproot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler

from core import DATA_FOLDER, RESULTS_FOLDER
from scripts.connie_training_utils import (Seed, extract_skeleton_features,
                                           extract_fourier_descriptors)

In [2]:
train_data = os.path.join(DATA_FOLDER, "train_data_root_full")
test_data = os.path.join(DATA_FOLDER, "test_data_root")

seed = Seed()
categories = ["Blob", "Diffusion Hit", "Electron", "Muon", "Others"]
#categories = ["Alpha", "Blob", "Diffusion_Hit", "Electron", "Muon", "Others"]

branch_name = "hitSumm"

In [3]:
train_all_data_list = []
test_all_data_list = []


print("Starting data loading")
for category in categories:
    train_category_path = os.path.join(train_data, category)
    train_root_files = glob(os.path.join(train_category_path, "*.root"))

    if not train_root_files:
        print(f"Warning: No .root files found in {train_category_path}")
        continue

    test_category_path = os.path.join(test_data, category)
    test_root_files = glob(os.path.join(test_category_path, "*.root"))

    if not test_root_files:
        print(f"Warning: No .root files found in {test_category_path}")
        continue

    print(f"TRAIN - Processing category: {category} ({len(train_root_files)} files)")
    for idx, file_path in enumerate(train_root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                train_file_branch = file[branch_name]
                df_train = train_file_branch.arrays(library="pd")
                df_train['label'] = category
                train_all_data_list.append(df_train)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

    print(f"TEST - Processing category: {category} ({len(test_root_files)} files)")
    for idx, file_path in enumerate(test_root_files):
        try:
            with uproot.open(file_path) as file:
                if branch_name not in file:
                    print(f"Warning: TTree '{branch_name}' not found in {file_path}. Skipping.")
                    continue
                test_file_branch = file[branch_name]
                df_test = test_file_branch.arrays(library="pd")
                df_test['label'] = category
                test_all_data_list.append(df_test)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")

Starting data loading
TRAIN - Processing category: Blob (351 files)
TEST - Processing category: Blob (62 files)
TRAIN - Processing category: Diffusion Hit (44 files)
TEST - Processing category: Diffusion Hit (8 files)
TRAIN - Processing category: Electron (366 files)
TEST - Processing category: Electron (65 files)
TRAIN - Processing category: Muon (2596 files)
TEST - Processing category: Muon (458 files)
TRAIN - Processing category: Others (220 files)
TEST - Processing category: Others (39 files)


Combine all DataFrames into a single DataFrame

In [4]:
if train_all_data_list:
    train_df_combined = pd.concat(train_all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(train_df_combined)} rows of data.")
else:
    print("No data loaded.")

if test_all_data_list:
    test_df_combined = pd.concat(test_all_data_list, ignore_index=True)
    print(f"Successfully loaded {len(test_df_combined)} rows of data.")
else:
    print("No data loaded.")

Successfully loaded 3577 rows of data.
Successfully loaded 632 rows of data.


In [5]:
train_df_processed = train_df_combined.copy()

train_df_processed["ePixMean"] = train_df_processed["ePix"].apply(np.mean)
train_df_processed["levelMean"] = train_df_processed["level"].apply(np.mean)


test_df_processed = test_df_combined.copy()

test_df_processed["ePixMean"] = test_df_processed["ePix"].apply(np.mean)
test_df_processed["levelMean"] = test_df_processed["level"].apply(np.mean)



In [6]:
skeleton_features_train = train_df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1
)

fd_features_train = train_df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)

skeleton_features_df_train = pd.json_normalize(skeleton_features_train)
fd_features_df_train = pd.json_normalize(fd_features_train)

train_df_processed = pd.concat([
    train_df_processed.reset_index(drop=True),
    skeleton_features_df_train,
    fd_features_df_train
], axis=1)

# Test data
skeleton_features_test = test_df_processed.apply(
    lambda row: extract_skeleton_features(row['xPix'],
                                          row['yPix']),
    axis=1)

fd_features_test = test_df_processed.apply(
    lambda row: extract_fourier_descriptors(row['xPix'],
                                            row['yPix'], 10),
    axis=1)

skeleton_features_df_test = pd.json_normalize(skeleton_features_test)
fd_features_df_test = pd.json_normalize(fd_features_test)

test_df_processed = pd.concat([
    test_df_processed.reset_index(drop=True),
    skeleton_features_df_test,
], axis=1)

test_df_processed = pd.concat([
    test_df_processed.reset_index(drop=True),
    fd_features_df_test
], axis=1)


In [7]:
train_df_processed = train_df_processed.drop(columns=["label", "xPix", "yPix",
                                                      "level", "ePix", "flag"])

# Drop columns with no variance
train_df_processed = train_df_processed.loc[:, train_df_processed.nunique() > 1]


test_df_processed = test_df_processed.drop(columns=["label", "xPix", "yPix",
                                                  "level", "ePix", "flag"])

# Drop columns with no variance
test_df_processed = test_df_processed.loc[:, test_df_processed.nunique() > 1]


Removing features from the dataframe

In [8]:
train_df_processed_final = train_df_processed.drop(columns=[
    "ohdu", "chid", "skpID", "runID", "imgID", "Gain", "expoStart",
    "DeltaT", "NpixAC", "E1", "n1", "xBary1", "yBary1",
    "xVar1", "yVar1", "nSavedPix", "nxPix", "nyPix",
    "nlevel", "nePix", "xMin", "xMax", "yMin", "yMax",
    "skeleton_length", "branch_to_end_ratio"
])

test_df_processed_final = test_df_processed.drop(columns=[
    "ohdu", "chid", "skpID", "runID", "imgID", "Gain", "expoStart",
    "DeltaT", "NpixAC", "E1", "n1", "xBary1", "yBary1",
    "xVar1", "yVar1", "nSavedPix", "nxPix", "nyPix",
    "nlevel", "nePix", "xMin", "xMax", "yMin", "yMax",
    "skeleton_length", "branch_to_end_ratio"
])


In [9]:
label_encoder = LabelEncoder()

X_train_split = train_df_processed_final.copy()
X_test_split = test_df_processed_final.copy()

scaler = StandardScaler()
X_train_split = scaler.fit_transform(X_train_split)
X_test_split = scaler.transform(X_test_split)

y_train_split = label_encoder.fit_transform(train_df_combined["label"])
y_test_split = label_encoder.transform(test_df_combined["label"])

for i, class_name in enumerate(label_encoder.classes_):
    print(f"Class ID {i}: {class_name}")


Class ID 0: Blob
Class ID 1: Diffusion Hit
Class ID 2: Electron
Class ID 3: Muon
Class ID 4: Others


In [34]:
best_params_per_class_xgb = {
    # Trial 19
    "Blob": {
        "eval_metric": "logloss",
        "gamma": 1.9977268790892988,
        "learning_rate": 0.19474630793576608, 
        "max_depth": 3,
        "n_estimators": 199,
        "random_state": 42,
        "use_label_encoder": False
    },
    # Trial 96
    "Diffusion Hit": {
        "eval_metric": "logloss",
        "gamma": 4.472478760324197,
        "learning_rate": 0.15483516469516256,
        "max_depth": 4,
        "n_estimators": 157,
        "random_state": 42,
        "use_label_encoder": False
    },
    # Trial 54 (35 ok, trial 88 pior, testando 54)
    "Electron": {
        "eval_metric": "logloss",
        "gamma": 2.781552955446174, #4.457331815810404,
        "learning_rate": 0.01706993635662494, #0.057821305295300385,
        "max_depth": 6, #4,
        "n_estimators": 200, #185,
        "random_state": 42,
        "use_label_encoder": False
    },
    # Trial 55
    "Muon": {
        "eval_metric": "logloss",
        "gamma": 0.32345324788309643,
        "learning_rate": 0.09796191207359622,
        "max_depth": 3,
        "n_estimators": 229,
        "random_state": 42,
        "use_label_encoder": False
    },
    # Trial 43
    "Others": {
        "eval_metric": "logloss",
        "gamma": 3.8266433711230143,
        "learning_rate": 0.27045757161175493,
        "max_depth": 4,
        "n_estimators": 331,
        "random_state": 42,
        "use_label_encoder": False
    }
}


In [11]:
best_params_per_class_rf = {
    # Trial 42
    "Blob": {
        "class_weight": "balanced_subsample",
        "max_features": 0.44372761547374784,
        "min_samples_leaf": 2,
        "min_samples_split": 6,
        "n_estimators": 125,
        "random_state": 42
    },
    # Trial 99
    "Diffusion Hit": {
        "class_weight": "balanced_subsample",
        "max_features": 0.3857661139450104,
        "min_samples_leaf": 9,
        "min_samples_split": 14,
        "n_estimators": 500,
        "random_state": 42
    },
    # Trial 81
    "Electron": {
        "class_weight": "balanced_subsample",
        "max_features": 0.382554632445477,
        "min_samples_leaf": 3,
        "min_samples_split": 28,
        "n_estimators": 451,
        "random_state": 42
    },
    # Trial 98
    "Muon": {
        "class_weight": "balanced_subsample",
        "max_features": 0.42236791418655806,
        "min_samples_leaf": 2,
        "min_samples_split": 7,
        "n_estimators": 430,
        "random_state": 42
    },
    # Trial 55
    "Others": {
        "class_weight": "balanced_subsample",
        "max_features": 0.18046955172823542,
        "min_samples_leaf": 2,
        "min_samples_split": 27,
        "n_estimators": 227,
        "random_state": 42
    }
}

In [12]:
def model_training(is_xgboost=True):
    models = {}
    if is_xgboost:
        best_params_per_class = best_params_per_class_xgb
        model_used = XGBClassifier
    else:
        best_params_per_class = best_params_per_class_rf
        model_used = RandomForestClassifier
    for class_name, params in best_params_per_class.items():
        class_id = label_encoder.transform([class_name])[0]
        y_binary = (y_train_split == class_id).astype(int)
        model = model_used(**params)
        if is_xgboost:
            pos_count = np.sum(y_binary == 1)
            neg_count = np.sum(y_binary == 0)
            scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
            model = model_used(**params, scale_pos_weight=scale_pos_weight)
        else:
            model = model_used(**params)
        model.fit(X_train_split, y_binary)
        models[class_name] = model
    return models

In [13]:
def ensemble_predict(models, X, label_encoder):
    probas = []
    for class_name in label_encoder.classes_:
        model = models[class_name]
        p = model.predict_proba(X)[:, 1]  # probability of positive class
        probas.append(p)
    probas = np.array(probas).T  # shape: (n_samples, n_classes)
    pred_indices = np.argmax(probas, axis=1)
    preds = label_encoder.classes_[pred_indices]
    return preds, probas

In [14]:
# Save metrics
metrics_dir_xgb = os.path.join(RESULTS_FOLDER, "test_metrics_xgboost_ova_img_descriptor_new_features")
metrics_dir_rf = os.path.join(RESULTS_FOLDER, "test_metrics_rf_ova_img_descriptor_new_features")
os.makedirs(metrics_dir_xgb, exist_ok=True)
os.makedirs(metrics_dir_rf, exist_ok=True)

Ensemble

In [15]:
def ensemble_models(is_xgboost=True):
    if is_xgboost:
        models = model_training()
        metrics_dir = metrics_dir_xgb
    else:
        models = model_training(is_xgboost=False)
        metrics_dir = metrics_dir_rf
    
    preds_str, probas = ensemble_predict(models, X_test_split, label_encoder)
    preds_int = label_encoder.transform(preds_str)
    
    report = classification_report(
        y_test_split,
        preds_int,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).transpose()
    
    cm = confusion_matrix(y_test_split,
                          preds_int,
                          labels=np.arange(len(label_encoder.classes_)))
    
    report_path = os.path.join(metrics_dir, "ova_classification_report.csv")
    report_df.to_csv(report_path)
    
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    cm_path = os.path.join(metrics_dir, "ova_confusion_matrix.png")
    fig.savefig(cm_path)
    plt.close(fig)

    cm_norm = confusion_matrix(
        y_test_split,
        preds_int,
        labels=np.arange(len(label_encoder.classes_)),
        normalize='true'
    )
    
    fig_norm, ax_norm = plt.subplots(figsize=(8,6))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
        ax=ax_norm
    )
    
    ax_norm.set_xlabel("Predicted")
    ax_norm.set_ylabel("True")
    #ax_norm.set_title("Normalized Confusion Matrix")
    
    fig_norm.tight_layout()
    
    cm_norm_path = os.path.join(
        metrics_dir,
        "ova_confusion_matrix_normalized.png"
    )
    
    fig_norm.savefig(cm_norm_path, dpi=300, bbox_inches='tight')
    plt.close(fig_norm)
    
    return models, report_df, cm, probas

In [35]:
# XGBoost OVA
(xgb_ova_models, xgb_ova_report,
 xgb_ova_cm, xgb_ova_probas) = ensemble_models(is_xgboost=True)

print("\n" + "="*60)
print("XGBoost OVA Ensemble Test Results")
print("="*60)
print(f"Accuracy:    {xgb_ova_report.loc['accuracy', 'precision']:.4f}")
print(f"Precision:   {xgb_ova_report.loc['macro avg', 'precision']:.4f}")
print(f"Recall:      {xgb_ova_report.loc['macro avg', 'recall']:.4f}")
print(f"Macro F1:    {xgb_ova_report.loc['macro avg', 'f1-score']:.4f}")


XGBoost OVA Ensemble Test Results
Accuracy:    0.9130
Precision:   0.8164
Recall:      0.8701
Macro F1:    0.8410


In [18]:
# RandomForest OVA
(rf_ova_models, rf_ova_report,
 rf_ova_cm, rf_ova_probas) = ensemble_models(is_xgboost=False)

print("\n" + "="*60)
print("RandomForest OVA Ensemble Test Results")
print("="*60)
print(f"Accuracy:    {rf_ova_report.loc['accuracy', 'precision']:.4f}")
print(f"Precision:   {rf_ova_report.loc['macro avg', 'precision']:.4f}")
print(f"Recall:      {rf_ova_report.loc['macro avg', 'recall']:.4f}")
print(f"Macro F1:    {rf_ova_report.loc['macro avg', 'f1-score']:.4f}")


RandomForest OVA Ensemble Test Results
Accuracy:    0.9130
Precision:   0.8091
Recall:      0.8444
Macro F1:    0.8241
